# TRUST-TC interval analysis quickstart

This notebook shows how to convert apparent thermal response test (TRT) estimates into uncertainty intervals and design reference factors with TRUST-TC.

## What TRUST-TC does

TRUST-TC starts from apparent thermal properties reported by a TRT interpretation model. It applies calibrated transformation uncertainty factors and returns a corrected median, a P05--P95 uncertainty interval, a reliability class, and a design reference factor.

Model labels used in this notebook are defined once here: LT-ILS is the late time infinite line source approximation, ILS is the exact infinite line source solution, ICS is the infinite cylindrical source or finite radius solution, and FLS is the finite line source solution.

## Setup

Run this notebook from the repository root. The path setup below also works when the notebook is launched from the `notebooks` folder.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src" / "trust_tc").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from trust_tc import run_interval_mode

OUTPUT_DIR = ROOT / "outputs" / "tool_demo"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ROOT

## Load an example input table

The stable input mode reads a CSV table with five required columns: `site`, `depth_interval`, `interpretation_model`, `parameter`, and `apparent_estimate`.

In [ ]:
input_csv = ROOT / "examples" / "example_fitted_results.csv"
example_input = pd.read_csv(input_csv)
example_input

## Run TRUST-TC interval mode

The function below writes `outputs/tool_demo/notebook_intervals.csv` and also returns the same table as a pandas DataFrame.

In [ ]:
notebook_output = OUTPUT_DIR / "notebook_intervals.csv"
result = run_interval_mode(
    input_csv,
    notebook_output,
    root=ROOT,
    calibration_domain="full_3d_production",
)

display_columns = [
    "site",
    "depth_interval",
    "interpretation_model",
    "parameter",
    "apparent_estimate",
    "corrected_median",
    "interval_p05",
    "interval_p95",
    "reliability_class",
    "design_reference_factor",
    "warning_flag",
]
result[display_columns]

## Read the interval output

`corrected_median` is the P50 interval estimate after applying the calibrated correction factor. `interval_p05` and `interval_p95` define the uncertainty interval. `reliability_class` is a reporting class from the calibration table and input diagnostics. `design_reference_factor` summarizes the design margin implied by the interval.

In [ ]:
result[[
    "parameter",
    "correction_factor_p05",
    "correction_factor_p50",
    "correction_factor_p95",
    "calibration_domain",
    "calibration_version",
]].drop_duplicates()

## Plot interval estimates

The plot below compares the apparent conductivity estimate with the corrected median and P05--P95 interval. It is a quick diagnostic plot, not a publication figure.

In [ ]:
import matplotlib.pyplot as plt

plot_data = result[result["parameter"].eq("lambda")].copy().reset_index(drop=True)
plot_data["label"] = plot_data["site"] + " | " + plot_data["depth_interval"].astype(str) + " | " + plot_data["interpretation_model"]

fig, ax = plt.subplots(figsize=(8, 3.8))
y = range(len(plot_data))
ax.hlines(y, plot_data["interval_p05"], plot_data["interval_p95"], color="0.25", linewidth=2, label="P05--P95 interval")
ax.scatter(plot_data["apparent_estimate"], y, marker="x", s=70, color="#333333", label="apparent estimate")
ax.scatter(plot_data["corrected_median"], y, marker="o", s=65, color="#1f77b4", label="corrected median")
ax.set_yticks(list(y))
ax.set_yticklabels(plot_data["label"])
ax.set_xlabel("thermal conductivity (W m$^{-1}$ K$^{-1}$)")
ax.set_title("TRUST-TC interval estimates for conductivity")
ax.grid(axis="x", color="0.9")
ax.legend(loc="best")
fig.tight_layout()

## Run field-style examples

The public CSV examples mimic common TRT reporting formats. This section uses `examples/ntu_tool_demo.csv` and `examples/fukuoka_tool_demo.csv`. They demonstrate interval calculation from apparent estimates; they are not raw field data.

In [ ]:
example_files = {
    "NTU public demo": ROOT / "examples" / "ntu_tool_demo.csv",
    "Fukuoka public demo": ROOT / "examples" / "fukuoka_tool_demo.csv",
}

summaries = []
for name, path in example_files.items():
    out_path = OUTPUT_DIR / f"{path.stem}_notebook_intervals.csv"
    table = run_interval_mode(path, out_path, root=ROOT, calibration_domain="full_3d_production")
    summaries.append({
        "example": name,
        "rows": len(table),
        "parameters": ", ".join(sorted(table["parameter"].unique())),
        "models": ", ".join(sorted(table["interpretation_model"].unique())),
        "output": str(out_path.relative_to(ROOT)),
    })

pd.DataFrame(summaries)

## Use a custom calibration table

Users can supply a calibration table explicitly. The bundled table, `src/trust_tc/calibration/full3d_production_correction_factor_distribution.csv`, is used below to show the required argument.

In [ ]:
calibration_csv = ROOT / "src" / "trust_tc" / "calibration" / "full3d_production_correction_factor_distribution.csv"
custom_output = OUTPUT_DIR / "notebook_custom_intervals.csv"
custom_result = run_interval_mode(
    input_csv,
    custom_output,
    root=ROOT,
    calibration_csv=calibration_csv,
    calibration_domain="full_3d_production",
)
custom_result[["site", "interpretation_model", "parameter", "corrected_median", "calibration_source"]].head()

## Command-line equivalent

The command-line interface produces the same type of interval table:

```powershell
python -m trust_tc interval --input examples/example_fitted_results.csv --output outputs/tool_demo/intervals.csv
```

Use the Python API when you want to integrate TRUST-TC into another analysis script. Use the command-line interface when you want a direct CSV-to-CSV conversion.